### Experiment: Effect of Prediction Head Capacity

In this experiment we study how the capacity of the prediction head affects performance when the temporal context is limited.

We use the same **nonlinear sequence simulation** described earlier. The sequence contains community-structured transitions where the traversal direction depends on the **parity of the sum of the previous \(K\) community visits**, introducing a long-range dependency.

The recurrent backbone is trained with a **fixed BPTT length of 3**, which is intentionally shorter than the dependency length. As a result, the recurrent layer alone cannot fully capture the required temporal structure.

To investigate whether downstream model capacity can compensate for this limitation, we gradually increase the **number of neurons in the prediction head** (MLP head) while keeping all other components fixed.

#### Experimental Setup

- Dataset: nonlinear community traversal sequence  
- Recurrent backbone: RNN  
- BPTT length: **3**  
- Training regime: single-pass online learning  
- Prediction head: MLP with varying hidden size  

#### Variable

We vary the **number of neurons in the prediction head**:


In [3]:
## Load necessary library files ##

import sys
sys.path.append('..')
from source.utils import get_sequence, DatasetConverter
from source.model.model import Model

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch import from_numpy as tnsr
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch.nn.functional as F
from tqdm import tqdm
import pickle 

In [4]:
## select device ##
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")  # works only with NVIDIA GPUs (not on Mac)
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [5]:
## load the nonlinear simulation from source files ##
print("A 42 tokens long nonlinear sequence ", get_sequence(42, n_community=2, n_members=3, context_depth=4))

A 42 tokens long nonlinear sequence  ABCGDEFGDEFGABCGDEFGDFEGDFEGDFEGDEFGDEFGDE


In [ ]:
total_samples, n_community, n_members, context_depth = 5000000, 2, 3, 4
total_layers, short_term_memory = 2, 4

vocab_size = 27
